# Qwen3 Embedding + Qwen3 Reranker + Qdrant — Kaggle Production Demo

This notebook demonstrates the two-model production-style retrieval path with the canonical 20K Qdrant snapshot. Use **Run All** on CPU. K=5 is the published retrieval default; the final cell verifies the documented runtime, semantic, and OOM qualification gates.


In [ ]:
# PRODUCTION DEMO HARDWARE / TOP-K CONFIG
import os, time
RUN_ALL_STARTED = time.perf_counter()
RUN_ALL_BUDGET_SECONDS = 600

def read_memory_events():
    from pathlib import Path
    path = Path('/sys/fs/cgroup/memory.events')
    out = {}
    if path.exists():
        for line in path.read_text().splitlines():
            key, value = line.split()
            out[key] = int(value)
    return out
MEMORY_EVENTS_BEFORE = read_memory_events()

RETRIEVAL_TOP_K = 5
RERANK_TOP_K = 5
DISPLAY_TOP_K = 5
LLAMA_SERVER_THREADS = 2
TORCH_NUM_THREADS = 2

# Example stronger host (replace values above):
# RETRIEVAL_TOP_K = 50
# RERANK_TOP_K = 50
# DISPLAY_TOP_K = 10
# LLAMA_SERVER_THREADS = 8
# TORCH_NUM_THREADS = 8

assert 1 <= DISPLAY_TOP_K <= RERANK_TOP_K <= RETRIEVAL_TOP_K
assert RERANK_TOP_K <= 200
for k, v in {
    "RETRIEVAL_TOP_K": RETRIEVAL_TOP_K,
    "RERANK_TOP_K": RERANK_TOP_K,
    "DISPLAY_TOP_K": DISPLAY_TOP_K,
    "LLAMA_SERVER_THREADS": LLAMA_SERVER_THREADS,
    "TORCH_NUM_THREADS": TORCH_NUM_THREADS,
    "MAX_RERANK_DOCUMENTS": RERANK_TOP_K,
    "TORCH_NUM_INTEROP_THREADS": 1,
    "MAX_CONCURRENT_INFERENCE": 1,
}.items(): os.environ[k] = str(v)
print({k: os.environ[k] for k in ["RETRIEVAL_TOP_K","RERANK_TOP_K","DISPLAY_TOP_K","LLAMA_SERVER_THREADS","TORCH_NUM_THREADS"]})


In [ ]:
# Locate this extracted source tree and install it editable.
from pathlib import Path
import subprocess, sys
candidates = [Path.cwd()] + list(Path('/kaggle/working').glob('**/pyproject.toml'))
APP_ROOT = None
for candidate in candidates:
    root = candidate if candidate.is_dir() else candidate.parent
    if (root/'src/qwen_dual_server').is_dir() and (root/'scripts/setup_llama_cpp_b10699.sh').is_file():
        APP_ROOT = root; break
assert APP_ROOT, 'Extract the source package under /kaggle/working first.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(APP_ROOT)], check=True)
print('APP_ROOT=', APP_ROOT)


In [ ]:
# Pin/setup Qdrant 1.18.3 and llama.cpp b10699.
import subprocess, os
q = subprocess.run(['bash', str(APP_ROOT/'scripts/setup-qdrant-production-demo.sh')], text=True, capture_output=True, check=True)
print(q.stdout)
for line in q.stdout.splitlines():
    if line.startswith('QDRANT_BIN='): os.environ['QDRANT_BIN'] = line.split('=',1)[1]
l = subprocess.run(['bash', str(APP_ROOT/'scripts/setup_llama_cpp_b10699.sh')], text=True, capture_output=True, check=True)
print(l.stdout)
for line in l.stdout.splitlines():
    if line.startswith('LLAMA_SERVER_BIN='): os.environ['LLAMA_SERVER_BIN'] = line.split('=',1)[1]
assert os.environ.get('QDRANT_BIN') and os.environ.get('LLAMA_SERVER_BIN')


In [ ]:
# Locate exact model/snapshot inputs. No model weights or snapshot are copied to /kaggle/working.
import hashlib, json
embedding_default = Path('/kaggle/input/models/dangkhoa2016/qwen-qwen3-embedding-4b/transformers/default/1')
assert embedding_default.is_dir(), 'Attach the Qwen3-Embedding-4B Kaggle model or set EMBEDDING_MODEL_PATH manually.'
os.environ['EMBEDDING_MODEL_PATH'] = str(embedding_default)

ggufs = list(Path('/kaggle/input').rglob('Qwen3-Reranker-4B.Q4_K_M.gguf'))
assert len(ggufs) == 1, f'Expected exactly one Q4_K_M GGUF, found {len(ggufs)}'
gguf_sha = hashlib.sha256(ggufs[0].read_bytes()).hexdigest()
assert gguf_sha == '941f7d1d1524251c026a797b803ac9575545c5d7aa19b26e0e49661d7720af49'
os.environ['RERANKER_GGUF_PATH'] = str(ggufs[0])

loc = subprocess.run([sys.executable, str(APP_ROOT/'scripts/locate-canonical-qdrant-snapshot.py')], text=True, capture_output=True, check=True)
snapshot = json.loads(loc.stdout)['path']
os.environ['QDRANT_BACKUP'] = snapshot
print('Embedding model:', embedding_default)
print('GGUF:', ggufs[0])
print('Snapshot:', snapshot)


In [ ]:
# Start empty local Qdrant and restore the immutable canonical 20K collection snapshot.
os.environ['PRODUCTION_DEMO_RUN_ROOT'] = '/kaggle/working/qwen3-hybrid-qdrant-production-demo'
r = subprocess.run(['bash', str(APP_ROOT/'scripts/restore-canonical-qdrant-snapshot.sh')], text=True, capture_output=True, check=True, env=os.environ.copy())
print(r.stdout)


In [ ]:
# Start the qualified hybrid inference service (Embedding FP16 + Reranker Q4_K_M).
import secrets, subprocess, time, urllib.request
run_root = Path(os.environ['PRODUCTION_DEMO_RUN_ROOT'])
run_root.mkdir(parents=True, exist_ok=True)
os.environ['DUAL_API_KEY'] = secrets.token_hex(24)
os.environ.update({
    'RERANKER_BACKEND':'llama_cpp', 'MODEL_DTYPE':'float16', 'QUANTIZATION_MODE':'none',
    'SERVER_HOST':'127.0.0.1', 'SERVER_PORT':'8000', 'LLAMA_SERVER_PORT':'8081',
    'LLAMA_SERVER_CONTEXT_SIZE':'1024', 'MAX_RERANK_DOCUMENTS':str(RERANK_TOP_K),
    'HF_HUB_OFFLINE':'1', 'TRANSFORMERS_OFFLINE':'1', 'ALLOW_REMOTE_MODEL_DOWNLOAD':'0',
})
log = open(run_root/'hybrid-server.log', 'w')
hybrid = subprocess.Popen(['bash', str(APP_ROOT/'scripts/start-server.sh')], stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())
for _ in range(900):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/ready', timeout=3) as response:
            if response.status == 200: break
    except Exception: pass
    assert hybrid.poll() is None, (run_root/'hybrid-server.log').read_text()[-4000:]
    time.sleep(1)
else: raise RuntimeError('hybrid readiness timeout')
print('HYBRID_READY=PASS pid=', hybrid.pid)


In [ ]:
# Run EN / VI / cross-language production demo and show before-vs-after ranking.
import json, subprocess, pandas as pd
p = subprocess.run([sys.executable, str(APP_ROOT/'scripts/run-production-demo.py')], text=True, capture_output=True, check=True, env=os.environ.copy())
DEMO = json.loads(p.stdout)
for case in DEMO['cases']:
    print('
###', case['case_id'], '-', case['query'])
    before = [{
        'rank': i+1,
        'name_en': r.get('payload',{}).get('name_en'),
        'name_vi': r.get('payload',{}).get('name_vi'),
        'qdrant_score': r.get('score'),
    } for i,r in enumerate(case['retrieval'][:DISPLAY_TOP_K])]
    after = [{
        'rank': i+1,
        'name_en': r.get('payload',{}).get('name_en'),
        'name_vi': r.get('payload',{}).get('name_vi'),
        'qdrant_score': r.get('score'),
        'rerank_score': r.get('rerank_score'),
    } for i,r in enumerate(case['display'])]
    display(pd.DataFrame(before).style.set_caption('Qdrant retrieval — before reranking'))
    display(pd.DataFrame(after).style.set_caption('Qwen3 reranker — after reranking'))
print('ALL_EXPECTED_TOP1_PASS=', DEMO['all_expected_top1_pass'])


In [ ]:
# Final fresh-session gate: runtime budget + semantic cases + cgroup OOM deltas.
import json, time
elapsed = time.perf_counter() - RUN_ALL_STARTED
events_after = read_memory_events()
oom_delta = events_after.get('oom',0) - MEMORY_EVENTS_BEFORE.get('oom',0)
oom_kill_delta = events_after.get('oom_kill',0) - MEMORY_EVENTS_BEFORE.get('oom_kill',0)
qualification_gate = (
    elapsed <= RUN_ALL_BUDGET_SECONDS
    and DEMO['all_expected_top1_pass']
    and oom_delta == 0
    and oom_kill_delta == 0
)
summary = {
    'run_all_seconds': round(elapsed,3),
    'budget_seconds': RUN_ALL_BUDGET_SECONDS,
    'all_expected_top1_pass': DEMO['all_expected_top1_pass'],
    'oom_delta': oom_delta,
    'oom_kill_delta': oom_kill_delta,
    'qualification_pass': qualification_gate,
    'retrieval_default': RETRIEVAL_TOP_K,
}
print(json.dumps(summary, indent=2))
print('PRODUCTION_QUALIFICATION=PASS' if qualification_gate else 'PRODUCTION_QUALIFICATION=FAIL')
assert qualification_gate, summary
